# Crawl CampusNest and extract tutors/listings (Playwright async)

Target site: https://campusnest.streamlit.app/


In [ ]:
!pip -q install playwright pandas requests
!playwright install chromium

In [ ]:
import re
import time
from collections import deque
from urllib.parse import urljoin, urlparse, urldefrag

import pandas as pd
import requests
from playwright.async_api import async_playwright

BASE_URL = "https://campusnest.streamlit.app/"
MAX_PAGES = 60
DEFAULT_DELAY = 1.0


In [ ]:
def fetch_robots_txt(base_url: str) -> str:
    robots_url = urljoin(base_url, "robots.txt")
    r = requests.get(robots_url, timeout=20)
    r.raise_for_status()
    return r.text

def parse_robots_txt(robots_text: str, user_agent: str = "*"):
    disallow = []
    crawl_delay = None
    active_agent = None
    for raw in robots_text.splitlines():
        line = raw.strip()
        if not line or line.startswith("#"):
            continue
        key, _, val = line.partition(":")
        key = key.strip().lower()
        val = val.strip()
        if key == "user-agent":
            active_agent = val
        elif key == "disallow" and active_agent in (user_agent, "*"):
            if val:
                disallow.append(val)
        elif key == "crawl-delay" and active_agent in (user_agent, "*"):
            try:
                crawl_delay = float(val)
            except ValueError:
                pass
    return {"disallow": disallow, "crawl_delay": crawl_delay}

def normalise_url(base_url: str, href: str) -> str | None:
    if not href:
        return None
    abs_url = urljoin(base_url, href)
    abs_url, _ = urldefrag(abs_url)
    return abs_url

def is_same_site(url: str, base_url: str) -> bool:
    return urlparse(url).netloc == urlparse(base_url).netloc

def is_allowed_by_robots(url: str, disallow_paths: list[str]) -> bool:
    path = urlparse(url).path or "/"
    return not any(path.startswith(d) for d in disallow_paths)

async def extract_card_texts(page) -> list[str]:
    cards = page.locator("div.sh-card")
    out = []
    n = await cards.count()
    for i in range(n):
        txt = (await cards.nth(i).inner_text()).strip()
        if txt:
            out.append(txt)
    return out

def parse_tutor_card(card_text: str) -> dict | None:
    m_id = re.search(r"\bID:\s*(TUT-\d{3})\b", card_text)
    if not m_id:
        return None
    tutor_id = m_id.group(1)
    lines = [ln.strip() for ln in card_text.splitlines() if ln.strip()]
    title = lines[0]
    m = re.match(r"^(?P<name>.+?)\s*·\s*€(?P<price>\d+)\/h$", title)
    if not m:
        m = re.match(r"^(?P<name>.+?)\s*·\s*€(?P<price>\d+)", title)
    name = m.group("name").strip() if m else title
    price = int(m.group("price")) if m and m.groupdict().get("price") else None
    headline = None
    subjects = None
    languages = None
    for ln in lines[1:]:
        if ln.lower().startswith("subjects:"):
            subjects = ln.split(":", 1)[1].strip()
        elif ln.lower().startswith("languages:"):
            languages = ln.split(":", 1)[1].strip()
        elif ln.lower().startswith("id:"):
            continue
        else:
            if headline is None:
                headline = ln
    return {"id": tutor_id, "name": name, "price_eur_per_hour": price, "headline": headline,
            "subjects": subjects, "languages": languages}

def parse_listing_card(card_text: str) -> dict | None:
    m_id = re.search(r"\bID:\s*(LIST-\d{4})\b", card_text)
    if not m_id:
        return None
    listing_id = m_id.group(1)
    lines = [ln.strip() for ln in card_text.splitlines() if ln.strip()]
    title_line = lines[0]
    m = re.match(r"^(?P<title>.+?)\s*·\s*€(?P<rent>\d+)\/mo$", title_line)
    title = m.group("title").strip() if m else title_line
    rent = int(m.group("rent")) if m and m.groupdict().get("rent") else None
    listing_type = None
    area = None
    m2 = re.match(r"^(?P<type>.+?)\s+in\s+(?P<area>.+)$", title)
    if m2:
        listing_type = m2.group("type").strip()
        area = m2.group("area").strip()
    min_stay_months = None
    deposit_eur = None
    for ln in lines[1:]:
        if "min stay" in ln.lower():
            ms = re.search(r"min stay:\s*(\d+)\s*months", ln, flags=re.I)
            if ms:
                min_stay_months = int(ms.group(1))
            dep = re.search(r"deposit:\s*€(\d+)", ln, flags=re.I)
            if dep:
                deposit_eur = int(dep.group(1))
    return {"id": listing_id, "title": title, "type": listing_type, "area": area,
            "rent_eur_per_month": rent, "min_stay_months": min_stay_months, "deposit_eur": deposit_eur}


In [ ]:
async def crawl_and_extract(base_url: str, max_pages: int = 60):
    robots_text = fetch_robots_txt(base_url)
    rules = parse_robots_txt(robots_text)
    disallow = rules["disallow"]
    delay = rules["crawl_delay"] if rules["crawl_delay"] is not None else DEFAULT_DELAY

    frontier = deque([base_url])
    visited = set()
    tutors = {}
    listings = {}
    log_rows = []

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()

        while frontier and len(visited) < max_pages:
            url = frontier.popleft()
            if url in visited:
                continue
            if not is_same_site(url, base_url):
                continue
            if not is_allowed_by_robots(url, disallow):
                continue

            try:
                await page.goto(url, wait_until="networkidle", timeout=45000)
                await page.wait_for_timeout(800)

                # discover links
                links = page.locator("a")
                n = await links.count()
                for i in range(n):
                    a = links.nth(i)
                    href = await a.get_attribute("href")
                    nxt = normalise_url(base_url, href) if href else None
                    if not nxt:
                        continue
                    if not is_same_site(nxt, base_url):
                        continue
                    if nxt in visited:
                        continue
                    if not is_allowed_by_robots(nxt, disallow):
                        continue
                    frontier.append(nxt)

                # extract cards
                cards = await extract_card_texts(page)
                for c in cards:
                    t = parse_tutor_card(c)
                    if t and t["id"] not in tutors:
                        tutors[t["id"]] = t
                    l = parse_listing_card(c)
                    if l and l["id"] not in listings:
                        listings[l["id"]] = l

                visited.add(url)
                log_rows.append({"url": url, "queue_size": len(frontier)})

            except Exception as e:
                visited.add(url)
                log_rows.append({"url": url, "error": str(e), "queue_size": len(frontier)})

            time.sleep(delay)

        await browser.close()

    tutors_df = pd.DataFrame(list(tutors.values())).sort_values("id").reset_index(drop=True)
    listings_df = pd.DataFrame(list(listings.values())).sort_values("id").reset_index(drop=True)
    log_df = pd.DataFrame(log_rows)

    tutors_df.to_csv("campusnest_crawled_tutors.csv", index=False)
    listings_df.to_csv("campusnest_crawled_listings.csv", index=False)
    log_df.to_csv("campusnest_crawl_log.csv", index=False)

    return tutors_df, listings_df, log_df

tutors_df, listings_df, log_df = await crawl_and_extract(BASE_URL, max_pages=MAX_PAGES)
tutors_df.head(), listings_df.head(), log_df.head(), (len(tutors_df), len(listings_df), len(log_df))
